In [19]:
import catboost as cb
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
import xgboost as xgb

In [20]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [21]:
y_train = train_df['Survived'].astype(int)
df = pd.concat(
    [train_df.assign(is_train=1), test_df.assign(is_train=0, Survived=np.nan)],
    sort=False,
).reset_index(drop=True)

In [22]:
df['Surname'] = df['Name'].apply(lambda x: x.split(',')[0].strip())
df['FareRound'] = df['Fare'].fillna(df['Fare'].median()).round(2)
df['GroupID'] = df['Surname'] + '_' + df['FareRound'].astype(str)

In [23]:
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
title_mapping = {
    'Mr': 'Mr',
    'Miss': 'Miss',
    'Mrs': 'Mrs',
    'Master': 'Master',
    'Mlle': 'Miss',
    'Mme': 'Mrs',
    'Ms': 'Miss',
}
df['Title'] = df['Title'].map(title_mapping).fillna('Rare')

In [24]:
df['Age'] = df.groupby(['Pclass', 'Title'])['Age'].transform(
    lambda x: x.fillna(x.median())
)

In [25]:
df['IsWomanOrChild'] = (
    (df['Sex'] == 'female') | (df['Title'] == 'Master')
).astype(int)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

In [26]:
df['CleanTicket'] = (
    df['Ticket']
    .str.replace('.', '', regex=False)
    .str.replace('/', '', regex=False)
    .str.strip()
)
ticket_counts = df['CleanTicket'].value_counts()
df['TicketGroupSize'] = df['CleanTicket'].map(ticket_counts)
df['FarePerPerson'] = df['Fare'] / df['TicketGroupSize']

In [27]:
train_only = df[df['is_train'] == 1]
wc_train = train_only[train_only['IsWomanOrChild'] == 1]

ticket_wcg = {}
family_wcg = {}

In [28]:
for ticket, group in wc_train.groupby('CleanTicket'):
  if len(group) > 0:
    ticket_wcg[ticket] = group['Survived'].mean()

In [29]:
for fid, group in wc_train.groupby('GroupID'):
  if len(group) > 0:
    family_wcg[fid] = group['Survived'].mean()

In [30]:
df['Embarked'] = df['Embarked'].fillna('S')
df_encoded = pd.get_dummies(
    df, columns=['Sex', 'Embarked', 'Title', 'Pclass'], drop_first=True
)

In [31]:
cols_to_drop = [
    'PassengerId',
    'Name',
    'Ticket',
    'CleanTicket',
    'Cabin',
    'Surname',
    'GroupID',
    'FareRound',
    'is_train',
    'Survived',
    'IsWomanOrChild',
]
feature_cols = [c for c in df_encoded.columns if c not in cols_to_drop]

In [32]:
X_train_raw = df_encoded[df['is_train'] == 1][feature_cols].reset_index(
    drop=True
)
X_test_raw = df_encoded[df['is_train'] == 0][feature_cols].reset_index(
    drop=True
)


In [33]:
imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train_raw), columns=feature_cols
)
X_test_imp = pd.DataFrame(imputer.transform(X_test_raw), columns=feature_cols)

In [34]:
continuous_cols = [
    'Age',
    'Fare',
    'FarePerPerson',
    'FamilySize',
    'TicketGroupSize',
    'SibSp',
    'Parch',
]
scaler = RobustScaler()
X_train = X_train_imp.copy()
X_test = X_test_imp.copy()

In [35]:
X_train[continuous_cols] = scaler.fit_transform(X_train_imp[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test_imp[continuous_cols])

In [36]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros((len(X_train), 5))
test_preds = np.zeros((len(X_test), 5))

In [37]:
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
  X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
  X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]

  m1 = xgb.XGBClassifier(
      n_estimators=100,
      max_depth=2,
      learning_rate=0.03,
      reg_lambda=3.0,
      subsample=0.8,
      random_state=42,
      eval_metric='logloss',
  ).fit(X_tr, y_tr)
  m2 = lgb.LGBMClassifier(
      n_estimators=100,
      max_depth=2,
      learning_rate=0.03,
      reg_lambda=2.0,
      random_state=42,
      verbose=-1,
  ).fit(X_tr, y_tr)
  m3 = cb.CatBoostClassifier(
      iterations=150, depth=3, learning_rate=0.03, verbose=0, random_seed=42
  ).fit(X_tr, y_tr)
  m4 = GradientBoostingClassifier(
      n_estimators=100, learning_rate=0.03, max_depth=3, random_state=42
  ).fit(X_tr, y_tr)
  m5 = RandomForestClassifier(
      n_estimators=200, max_depth=3, min_samples_leaf=2, random_state=42
  ).fit(X_tr, y_tr)

  oof_preds[val_idx, 0] = m1.predict_proba(X_va)[:, 1]
  oof_preds[val_idx, 1] = m2.predict_proba(X_va)[:, 1]
  oof_preds[val_idx, 2] = m3.predict_proba(X_va)[:, 1]
  oof_preds[val_idx, 3] = m4.predict_proba(X_va)[:, 1]
  oof_preds[val_idx, 4] = m5.predict_proba(X_va)[:, 1]

  test_preds[:, 0] += m1.predict_proba(X_test)[:, 1] / 5.0
  test_preds[:, 1] += m2.predict_proba(X_test)[:, 1] / 5.0
  test_preds[:, 2] += m3.predict_proba(X_test)[:, 1] / 5.0
  test_preds[:, 3] += m4.predict_proba(X_test)[:, 1] / 5.0
  test_preds[:, 4] += m5.predict_proba(X_test)[:, 1] / 5.0

meta_model = LogisticRegression(C=0.5, random_state=42).fit(oof_preds, y_train)
raw_test_preds = meta_model.predict(test_preds)


In [38]:
test_df_out = df[df['is_train'] == 0].copy().reset_index(drop=True)
test_df_out['Pred'] = raw_test_preds


In [39]:
overrides = 0
for i, row in test_df_out.iterrows():
  # Priority 1: Match Clean Ticket; Priority 2: Match GroupID
  signal = ticket_wcg.get(
      row['CleanTicket'], family_wcg.get(row['GroupID'], None)
  )

  if signal is not None:
    # Rule 1: Females/children in 100% perishing groups -> 0
    if row['IsWomanOrChild'] == 1 and signal == 0.0:
      if test_df_out.loc[i, 'Pred'] != 0:
        test_df_out.loc[i, 'Pred'] = 0
        overrides += 1
    # Rule 2: Master/young boys in 100% surviving groups -> 1
    elif row['Title'] == 'Master' and signal == 1.0:
      if test_df_out.loc[i, 'Pred'] != 1:
        test_df_out.loc[i, 'Pred'] = 1
        overrides += 1

print(f'Applied {overrides} clean WCG post-processing overrides.')

Applied 8 clean WCG post-processing overrides.


In [40]:
submission = pd.DataFrame({
    'PassengerId': test_df_out['PassengerId'].astype(int),
    'Survived': test_df_out['Pred'].astype(int),
})

assert len(submission) == 418
assert list(submission.columns) == ['PassengerId', 'Survived']
assert submission.isnull().sum().sum() == 0

submission.to_csv('submission_clean_leakfree.csv', index=False)
print('Saved submission_clean_leakfree.csv successfully!')

Saved submission_clean_leakfree.csv successfully!
